# Mini GPT: Decoder-Only Transformer from Scratch

This notebook builds a complete **GPT-2 style autoregressive language model** from scratch using pure PyTorch.

### Key Architecture Components:
1. **Character-Level Tokenization**: Map characters to integers and vice-versa.
2. **Causal Multi-Head Self-Attention**: Self-attention with upper-triangular causal masking so tokens only attend to past context.
3. **Pre-LayerNorm & Residual Connections**: Stable gradient propagation matching GPT-2 / modern LLMs.
4. **Feed-Forward MLP**: 4x expansion with GELU non-linear activation.
5. **Autoregressive Sampling**: Sampling next tokens iteratively with temperature and top-$k$ filtering.


## 1. Imports & Hardware Configuration
We detect available hardware acceleration (`MPS` on Apple Silicon, `CUDA` on Nvidia GPUs, or fallback to `CPU`).


In [ ]:
import math
import time
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

# Device selection: Apple Silicon (mps) > Nvidia (cuda) > CPU
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"[*] Hardware accelerator: {device.upper()}")
torch.manual_seed(1337)

## 2. Hyperparameters
Define sequence context length, batch size, embedding dimension, attention heads, and layer count.


In [ ]:
batch_size = 64        # How many independent sequences to process in parallel
block_size = 128       # Maximum context length (tokens the model can look back)
max_iters = 1500       # Total training iterations (~1-2 minutes on Mac MPS)
eval_interval = 250    # How often to evaluate train/val loss
learning_rate = 3e-4   # AdamW learning rate
eval_iters = 100       # Number of batches to average for evaluation
n_embd = 192           # Embedding dimension (each token is a vector of this size)
n_head = 6             # Number of attention heads (n_embd // n_head = 32 dim per head)
n_layer = 4            # Number of Transformer blocks
dropout = 0.1          # Dropout rate for regularization

## 3. Dataset & Character-Level Tokenizer
Load the Tiny Shakespeare dataset (`input.txt`) and create character mappings.


In [ ]:
# Support running from within 'gemini' directory or repository root
input_path = Path("input.txt")
if not input_path.is_file():
    input_path = Path("gemini/input.txt")

with input_path.open("r", encoding="utf-8") as f:
    text = f.read()

# Build sorted vocabulary of unique characters
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"[*] Total dataset characters: {len(text):,}")
print(f"[*] Vocabulary size: {vocab_size} unique characters")
print(f"[*] Characters: {''.join(chars[:20])} ...")

# Mappings from characters to integers and vice-versa
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

# Split into 90% train, 10% validation
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"[*] Train tokens: {len(train_data):,}, Val tokens: {len(val_data):,}")

## 4. Batch Generation & Shifted Target Pairs
For each input sequence $X$, the target sequence $Y$ is $X$ shifted by 1 token to the right. The model learns to predict the next token at every position.


In [ ]:
def get_batch(split: str):
    """Generate a small batch of inputs X and target next-tokens Y."""
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model):
    """Estimate train and validation loss over multiple batches without gradient tracking."""
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# Preview one batch shape
xb, yb = get_batch("train")
print(f"[*] Input batch shape: {xb.shape} (batch_size, block_size)")
print(f"[*] Target batch shape: {yb.shape}")

## 5. Causal Multi-Head Self-Attention
Each attention head computes Query ($Q$), Key ($K$), and Value ($V$) projections.
A lower-triangular causal mask (`bias`) fills future positions with $-\infty$, ensuring that token $t$ cannot attend to tokens $t+1$ or later.


In [ ]:
class CausalSelfAttention(nn.Module):
    """Multi-Head Causal Self-Attention with upper-triangular masking."""

    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        # Key, Query, Value projections combined into a single linear projection (3 * n_embd)
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # Causal mask: lower-triangular matrix of 1s
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(block_size, block_size)).view(
                1, 1, block_size, block_size
            ),
        )

    def forward(self, x):
        B, T, C = x.size()

        # Calculate Q, K, V for all heads and transpose for matrix multiplication
        qkv = self.c_attn(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Scaled Dot-Product Attention: Softmax((Q @ K^T) / sqrt(d_k) + Mask) @ V
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # Output projection back to residual stream
        return self.resid_dropout(self.c_proj(y))

## 6. Feed-Forward MLP & Transformer Block
- **MLP**: A 2-layer network expanding hidden dimensions 4x with GELU activation.
- **Block**: Combines Pre-LayerNorm, Self-Attention, and MLP with residual skip connections: $x = x + \text{Attn}(\text{LN}(x))$. 


In [ ]:
class MLP(nn.Module):
    """Position-wise Feed-Forward Network: 4x expansion with GELU activation."""

    def __init__(self, n_embd: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer Block with Pre-LayerNorm and Residual Connections."""

    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        # Pre-LayerNorm formulation (GPT-2 style)
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

## 7. The Full MiniGPT Model
Stacks token embeddings, learned position embeddings, $N$ Transformer blocks, final LayerNorm, and a weight-tied linear prediction head.


In [ ]:
class MiniGPT(nn.Module):
    """Complete Decoder-Only Autoregressive Language Model."""

    def __init__(self):
        super().__init__()
        self.block_size = block_size

        # Token and Positional Embeddings
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)

        # Stack of Transformer blocks
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )

        # Final LayerNorm and Unembedding projection head
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying: share token embedding and output head weights
        self.tok_emb.weight = self.head.weight

        # Initialize weights with standard normal distribution (std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.block_size, f"Context length {T} exceeds block size {self.block_size}"

        # 1. Embed tokens and positions
        tok_embeddings = self.tok_emb(idx)
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_embeddings = self.pos_emb(pos)
        x = self.drop(tok_embeddings + pos_embeddings)

        # 2. Pass through Transformer blocks
        x = self.blocks(x)

        # 3. Final normalization and logits projection
        x = self.ln_f(x)
        logits = self.head(x)

        # 4. Compute Cross-Entropy Loss if targets are provided
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 0.8, top_k: int = 40):
        """Autoregressively sample new tokens."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

## 8. Initialize Model and Optimizer
Create the model and configure the AdamW optimizer with decoupled weight decay.


In [ ]:
model = MiniGPT().to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"[*] Total Model Parameters: {param_count:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-1)

## 9. Training Loop
Train the Transformer for 1,500 steps, tracking train and validation loss.


In [ ]:
print("=" * 55)
print("  Beginning Training")
print("=" * 55)
start_time = time.time()

for it in range(max_iters + 1):
    # Periodic evaluation
    if it % eval_interval == 0:
        losses = estimate_loss(model)
        elapsed = time.time() - start_time
        print(
            f"Step {it:4d}/{max_iters:4d} | "
            f"Train Loss: {losses['train']:.4f} | "
            f"Val Loss: {losses['val']:.4f} | "
            f"Elapsed: {elapsed:.1f}s"
        )

    # Sample batch and backpropagate
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

total_time = time.time() - start_time
print("=" * 55)
print(f"[*] Training finished in {total_time:.1f} seconds.")

## 10. Generate Text
Sample text from an unprompted starting context, or prompt it with custom starting text.


In [ ]:
# Unprompted generation (starts with a zero/newline token)
print("--- Unprompted Generation ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
out = model.generate(context, max_new_tokens=300, temperature=0.8, top_k=40)
print(decode(out[0].tolist()))

# Custom prompt generation
print("\n--- Custom Prompted Generation ---")
prompt = "JULIET:\nO Romeo, wherefore art thou "
prompt_encoded = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
prompt_out = model.generate(prompt_encoded, max_new_tokens=250, temperature=0.7, top_k=40)
print(decode(prompt_out[0].tolist()))